# Notebook 5.1: Reaction-Diffusion Transport

## Objective
Add chemistry to transport. First-order reaction (decay):
$$\frac{\partial c}{\partial t} + \mathbf{u} \cdot \nabla c = D \nabla^2 c - k c$$

where $k$ [s⁻¹] is the reaction rate. Examples: enzyme degradation, photobleaching, fluorophore quenching.

**Analytical check (no flow, no diffusion):** $c(t) = c_0 e^{-kt}$

In [ ]:
from fenics import *
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
set_log_level(LogLevel.WARNING)

---
## Step 1: Solve Poiseuille Flow

In [ ]:
L, H, mu, dp = 1.0, 0.1, 1.0, 1.0
mesh = RectangleMesh(Point(0.0, -H/2), Point(L, H/2), 60, 15)

W = FunctionSpace(mesh, MixedElement([
    VectorElement("P", mesh.ufl_cell(), 2),
    FiniteElement("P", mesh.ufl_cell(), 1)
]))

tol = 1e-10
(u, p) = TrialFunctions(W)
(v, q) = TestFunctions(W)

a_s = (mu * inner(grad(u), grad(v)) - p*div(v) + q*div(u)) * dx
L_s = dot(Constant((-dp/L, 0.0)), v) * dx
bc_s = DirichletBC(W.sub(0), Constant((0.0, 0.0)),
                   f"on_boundary && (x[1] > {H/2-tol} || x[1] < {-H/2+tol})")

w = Function(W)
solve(a_s == L_s, w, [bc_s])
u_flow, _ = w.split()

print(f"Poiseuille solved. U_max = {u_flow.vector().norm('linf'):.5f}")

---
## Step 2: Reaction-Diffusion Transport

Implicit Euler weak form:
$$\int\frac{c^{n+1}}{\Delta t} \phi \, dx + \int(\mathbf{u}\cdot\nabla c^{n+1})\phi \, dx + D\int\nabla c^{n+1}\cdot\nabla\phi \, dx + k\int c^{n+1}\phi \, dx = \int\frac{c^n}{\Delta t}\phi \, dx$$

The only change vs. passive transport: `+ k * c * phi * dx`

In [ ]:
D  = 0.002   # Diffusivity
k  = 1.0     # Reaction rate (try: 0, 0.5, 2, 5)
dt = 0.05
T  = 2.0

S   = FunctionSpace(mesh, "P", 1)
c   = TrialFunction(S)
phi = TestFunction(S)
c_n = Function(S)

bc_c = DirichletBC(S, Constant(1.0), f"on_boundary && x[0] < {tol}")

# Reaction term adds +k*c*phi*dx compared to pure transport
a_c = (c/dt * phi + dot(u_flow, grad(c)) * phi
       + D * dot(grad(c), grad(phi))
       + k * c * phi) * dx
L_c = c_n/dt * phi * dx

print(f"D = {D},  k = {k},  Da = k*L/U = {k*L/u_flow.vector().norm('linf'):.2f} (Damköhler number)")

---
## Step 3: Time Loop

In [ ]:
t = 0.0
snapshots = {}
save_times = [0.5, 1.0, 2.0]

A_c = assemble(a_c)
bc_c.apply(A_c)
c_sol = Function(S)

while t < T - 1e-8:
    t += dt
    b_c = assemble(L_c)
    bc_c.apply(b_c)
    solve(A_c, c_sol.vector(), b_c)
    c_n.assign(c_sol)
    for ts in save_times:
        if abs(t - ts) < dt/2 and ts not in snapshots:
            snapshots[ts] = c_sol.copy(deepcopy=True)
            total_mass = assemble(c_sol * dx)
            print(f"  t = {ts:.2f}, total mass = {total_mass:.5f}")

print("Done.")

---
## Step 4: Visualise & Compare Reaction Rates

In [ ]:
fig, axes = plt.subplots(len(snapshots), 1, figsize=(13, 2.5*len(snapshots)))
if len(snapshots) == 1: axes = [axes]

for ax, (ts, cs) in zip(axes, sorted(snapshots.items())):
    im = plot(cs, ax=ax, cmap='YlOrRd_r', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, label='$c$')
    ax.set_title(f't = {ts}  (k = {k})', fontsize=11)
    ax.set_ylabel('$y$')

axes[-1].set_xlabel('$x$')
plt.suptitle(f'Reactive tracer (D={D}, k={k})', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/reaction_diffusion.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary

| Term | FEniCS expression |
|------|-------------------|
| Time derivative | `c/dt * phi * dx` |
| Advection | `dot(u, grad(c)) * phi * dx` |
| Diffusion | `D * dot(grad(c), grad(phi)) * dx` |
| Reaction (decay) | `k * c * phi * dx` |

**Damköhler number** $Da = kL/U$: ratio of reaction to convection time.
- $Da \ll 1$: reaction negligible
- $Da \gg 1$: species consumed before reaching outlet

**Exercise:** Run with `k=0`, `k=1`, `k=5`. How does the outlet concentration change with $k$?